# Surprisal methods

The factory `surprisal_model()` exposes these output-layer conventions:

| Method | Class | Reference |
|---|---|---|
| `"surprisal"` | `HuggingFaceLM` | Wilcox et al. (2023), Pimentel et al. (2023), Giulianelli et al. (2023), Kuribayashi et al. (2021) |
| `"wordsprobability_surprisal_buggy"` | `WordsProbabilityLM` | Pimentel & Meister (2024) (standard convention) |
| `"wordsprobability_surprisal"` | `WordsProbabilityLM` | Pimentel & Meister (2024) (BOW/BOS/EOW correction) |
| `"char_beam"` | `CharBeamLM` | Giulianelli et al. (2024b) (top-K marginalisation) |
| `"modelblocks_surprisal"` | `ModelBlocksSurprisalLM` | ModelBlocks word-boundary convention, used by Clark et al. (2025) |

They give different numbers on the same input.

## Prerequisites

```bash
pip install -e ".[models,character-surprisal]" --group word-surprisal
```

With pip 25.1 or newer, the `word-surprisal` dependency group installs the pinned `wordsprobability` source that `WordsProbabilityLM` wraps, and `character-surprisal` installs `genlm-bytes` for `CharBeamLM`.

In [ ]:
import logging

import numpy as np

from replica.core import AnnotationLayer, Span, Stimulus
from replica.feature import surprisal_model

logging.getLogger("replica").setLevel(logging.WARNING)

## 1. Build a stimulus

A `Stimulus` is text + annotation layers. This example uses a `"words"` layer of whitespace tokens, the unit most papers report.

In [ ]:
sentence = "The quick brown fox jumps over the lazy dog"
words = sentence.split(" ")

spans, off = [], 0
for w in words:
    spans.append(Span(off, off + len(w)))
    off += len(w) + 1

stim = Stimulus(text=sentence, stimulus_id="example")
stim.add_layer(AnnotationLayer(name="words", spans=spans, values=words))
print(f"text   : {stim.text!r}")
print(f"units  : {len(stim.layers['words'])} words")

## 2. Standard convention: `"surprisal"` (`HuggingFaceLM`)

Per-word surprisal sums the negative log probabilities of the tokens assigned to the word. With `add_bos=True` (the default), scoring conditions on the resolved start token; model defaults or explicit `start_ids` select the prefix.

In [ ]:
standard = surprisal_model("gpt2", method="surprisal", add_bos=True)
out = standard(stim, target_layer="words")
standard_nats = -np.array([o.values["log_p_observed"] for o in out])
print(f"{'word':>10s}  {'surp (nats)':>12s}")
for w, s in zip(words, standard_nats, strict=False):
    print(f"{w!r:>10s}  {s:>12.4f}")

## 3. Pimentel & Meister 2024: buggy vs corrected

Pimentel & Meister (2024) identify that the standard convention mis-counts probability mass at BOW/BOS/EOW boundaries. The package exposes both:

- `"wordsprobability_surprisal_buggy"` (`correction="buggy"`): the standard convention. Agreement with `"surprisal"` requires the same tokenizer, prefix and context.
- `"wordsprobability_surprisal"` (`correction="corrected"`): their fix.

Both take the Hugging Face id `"gpt2"`, which maps to the `wordsprobability` package's `gpt2-small`.

In [ ]:
buggy = surprisal_model("gpt2", method="wordsprobability_surprisal_buggy")
corrected = surprisal_model("gpt2", method="wordsprobability_surprisal")

buggy_nats = -np.array(
    [o.values["log_p_observed"] for o in buggy(stim, target_layer="words")]
)
corr_nats = -np.array(
    [o.values["log_p_observed"] for o in corrected(stim, target_layer="words")]
)

print(f"{'word':>10s}  {'buggy':>10s}  {'corrected':>10s}  {'diff':>8s}")
for w, b, c in zip(words, buggy_nats, corr_nats, strict=False):
    print(f"{w!r:>10s}  {b:>10.4f}  {c:>10.4f}  {abs(b - c):>+8.4f}")

**Sanity check.** `buggy` should match the `surp (nats)` column from §2; `corrected` differs by the BOW/EOW correction magnitude.

In [ ]:
max_diff = np.max(np.abs(buggy_nats - standard_nats))
print(
    f"max |HuggingFaceLM(add_bos=True) - WordsProbabilityLM(buggy)| = {max_diff:.2e} nats"
)
assert max_diff < 1e-4, (
    "the two implementations of the standard convention should agree"
)

print(f"max |buggy - corrected| = {np.max(np.abs(buggy_nats - corr_nats)):.4f} nats")

## 4. Giulianelli et al. (2024b): tokenisation marginalisation

`CharBeamLM` approximates marginalisation over tokenisations using a beam of width `K`. It sums byte log-probabilities over each unit in the requested annotation layer. Here we request one output per character; word spans can also be scored directly.


In [ ]:
char_spans = [Span(i, i + 1) for i in range(len(sentence))]
char_stim = Stimulus(text=sentence, stimulus_id="example_chars")
char_stim.add_layer(
    AnnotationLayer(name="characters", spans=char_spans, values=list(sentence))
)

char_lm = surprisal_model("gpt2", method="char_beam", K=5)
char_out = char_lm(char_stim, target_layer="characters")
char_nats = -np.array([o.values["log_p_observed"] for o in char_out])

# Sum characters inside each word span; spaces are excluded here.
# This is not a partition of the complete string probability.
char_per_word = []
for w_span in spans:
    char_per_word.append(float(char_nats[w_span.start : w_span.end].sum()))

print(f"{'word':>10s}  {'standard':>10s}  {'char_beam':>11s}  {'diff':>8s}")
for w, std, cb in zip(words, standard_nats, char_per_word, strict=False):
    print(f"{w!r:>10s}  {std:>10.4f}  {cb:>11.4f}  {std - cb:>+8.4f}")

## 5. ModelBlocks word-boundary convention

`modelblocks_surprisal` computes surprisal through the local
`ModelBlocksSurprisalLM` backend; it does not require a ModelBlocks checkout.
The backend scores subword tokens with overlapping context windows, moves
word-boundary probability mass between adjacent tokens, and sums to words.
It returns `log_p_observed` in nats, as the other factory methods do.
Its sequence-edge convention differs from the corrected `wordsprobability`
method, so their complete output arrays need not agree.


In [ ]:
modelblocks = surprisal_model("gpt2", method="modelblocks_surprisal")
modelblocks_nats = -np.array(
    [o.values["log_p_observed"] for o in modelblocks(stim, target_layer="words")]
)
for word, value in zip(words, modelblocks_nats, strict=True):
    print(f"{word!r:>10s}  {value:>12.4f} nats")

For predictors already computed by ModelBlocks, `ModelBlocksFeature` reads
`.itemmeasures` tables instead. `ModelBlocksRunner` can build those tables from
an installed ModelBlocks checkout. These are separate interfaces from the
surprisal backend used above; see [`replica.feature.modelblocks`](../../src/replica/feature/modelblocks.py).


`char_beam` approximates a sum over tokenisations under one token vocabulary.
`surprisal` scores the tokenizer's selected tokenisation. Their word sums also
depend on which adjacent spaces are included.

## 6. Picking a method

- **`surprisal`**: sum subword scores using the selected tokenization and context policy.
- **`wordsprobability_surprisal`**: word-boundary-corrected probabilities.
- **`char_beam`**: character-level analysis or tokenisation-sensitive comparisons.
- **`modelblocks_surprisal`**: the ModelBlocks word-boundary and context-window convention.

Choose the convention required by the scientific question or study recipe.
Matching a method name alone does not establish a match in context, boundaries
or aggregation.

Each method returns a `FeatureModel`. Bind it to an extractor with `ModelFeature(model, Surprisal())`, then select that feature in `EvaluationSpec`. The raw log probabilities above are in nats; `Surprisal()` extracts bits by default.